## IMPORTATION DES BIBLIOTHEQUES ET PARAMETRES GLOBAUX

In [ ]:
# 1. Imports
import os
import time
import math
import importlib
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box, Point
from pyproj import CRS

# Recharger le module de fonctions annexes
import fonctions_annexes_biodiv
importlib.reload(fonctions_annexes_biodiv)
from fonctions_annexes_biodiv import generer_dictionnaire_taxonomie

# 2. Constantes et variables globales
%matplotlib qt
path_data = r'D:\MANTIS\Data'

# 4. Fonction pour générer le chemin de sauvegarde
def generate_file_path(country_name, grid_size_km, grid_type, cle_geo):
    return path_data + f"\\GBIF_{country_name.replace(' ', '_')}\\SIG\\grid_{country_name.replace(' ', '_')}_{grid_type}_{cle_geo}.geojson"


## OPERATION SUR LES DONNEES GLOBALES

In [ ]:
def load_geospatial_data(path_data):
    """
    Charge les fichiers SIG globaux et spécifiques à la France.

    Args:
        path_data (str): Chemin du répertoire contenant les fichiers SIG.

    Returns:
        dict: Un dictionnaire contenant les GeoDataFrames chargés.
    """
    sig_path = os.path.join(path_data, "SIG_global")
    france_path = os.path.join(path_data, "GBIF_France", "SIG")

    geodata = {
        "world_terrestre": gpd.read_file(os.path.join(sig_path, "world-administrative-boundaries.geojson")),
        "world_maritime": gpd.read_file(os.path.join(sig_path, "eez_v11.gpkg")),
        "departement_gpd": gpd.read_file(os.path.join(france_path, "carte_departements.geojson")),
        "PNR_gpd": gpd.read_file(os.path.join(france_path, "N_ENP_PNR_S_000.shx"))[['NOM_SITE', 'geometry']],
        "PN_gpd": gpd.read_file(os.path.join(france_path, "N_ENP_PN_S_000.shx"))[['NOM_SITE', 'geometry']]
    }

    return geodata

path_data = "chemin/vers/tes/données"  # Remplace par ton vrai chemin
geo_data = load_geospatial_data(path_data)

# Accéder aux données
world_terrestre = geo_data["world_terrestre"]
world_maritime = geo_data["world_maritime"]
departement_gpd = geo_data["departement_gpd"]
PNR_gpd = geo_data["PNR_gpd"]
PN_gpd = geo_data["PN_gpd"]



## GENERATION DE LA GRILLE POUR UN PAYS

In [47]:
# 3. Définition des fonctions
def degrees_per_km(latitude):
    """Calcule la conversion degrés/km pour une latitude donnée."""
    lat_deg_per_km = 1 / 111.32
    lon_deg_per_km = 1 / (111.32 * math.cos(math.radians(latitude)))
    return lat_deg_per_km, lon_deg_per_km

def create_country_grid_WGS84(gdf, country_code, col_code="color_code", grid_size_km=10, midpoint_lat=None, crop=False, display=False):
    """Génère une grille pour un pays spécifique en WGS84."""
    country = gdf[gdf[col_code] == country_code]
    if country.empty:
        raise ValueError(f"Pays '{country_code}' introuvable dans le GeoDataFrame.")

    def create_grid(country, grid_size, reference_point=(0, 0), midpoint_lat=None, crop=False):
        """Création d'une grille avec une taille définie."""
        ref_x, ref_y = reference_point
        country_geometry = country.geometry.union_all()
        minx, miny, maxx, maxy = country_geometry.bounds
        
        if midpoint_lat is None:
            midpoint_lat = (miny + maxy) / 2
        lat_deg_per_km, lon_deg_per_km = degrees_per_km(midpoint_lat)
        dy, dx = grid_size * lat_deg_per_km, grid_size * lon_deg_per_km
        start_x, start_y = ref_x + ((minx - ref_x) // dx) * dx, ref_y + ((miny - ref_y) // dy) * dy
        
        grid_cells = []
        for x in np.arange(start_x, maxx, dx):
            for y in np.arange(start_y, maxy, dy):
                cell = box(x, y, x + dx, y + dy)
                grid_cells.append({"geometry": cell, "min_lon": x, "min_lat": y, "max_lon": x + dx, "max_lat": y + dy})
        
        grid_gdf = gpd.GeoDataFrame(grid_cells, crs=country.crs)
        return gpd.clip(grid_gdf, country_geometry) if crop else grid_gdf
                

    grid = create_grid(country, grid_size_km, midpoint_lat=midpoint_lat, crop=crop)
    grid = grid[grid.intersects(country.geometry.union_all())]
    grid["cell_name"] = grid.apply(lambda cell: f"{grid_size_km}kmE{int(abs(cell.geometry.centroid.x) * 100):05d}N{int(abs(cell.geometry.centroid.y) * 100):05d}{country_code}", axis=1)
    grid["country_code"] = country_code

    if display:
        fig, ax = plt.subplots(figsize=(10, 10))
        gdf.plot(ax=ax, edgecolor="black", linewidth=0.5)
        country.plot(ax=ax, edgecolor="red", linewidth=2, facecolor="none")
        grid.plot(ax=ax, color="lightblue", edgecolor="grey", alpha=0.6)
        plt.show()

    return grid

def check_duplicates(grid, grid_name, cle_geo):
    if not grid.empty:
        duplicate_count = grid[cle_geo].duplicated().sum()
        if duplicate_count > 0:
            print(f"⚠️ Attention : {duplicate_count} doublon(s) trouvé(s) dans {grid_name} !")
        else:
            print(f"✅ Aucun doublon trouvé dans {grid_name}.")

def make_geo_keys_unique(grid, cle_geo):
    if not grid.empty:
        grid["cle_geo_unique"] = grid.groupby(cle_geo).cumcount().astype(str)
        grid.loc[grid["cle_geo_unique"] != "0", cle_geo] += "_" + grid["cle_geo_unique"]
        grid.drop(columns=["cle_geo_unique"], inplace=True)  # Supprime la colonne temporaire



In [56]:
# Sélection du pays et création de la grille
country_name = "France"
grid_size_km = 20
cle_geo = f"codeMaille{grid_size_km}Km"

country_code = world_terrestre[world_terrestre["name"] == country_name]["color_code"].iloc[0]
print(country_code)

ESP


In [57]:
critere_terrestre="color_code"
critere_maritime="ISO_TER1"

# 1. Sélection du pays terrestre et maritime
#country_terrestre = world_terrestre[world_terrestre[critere_terrestre] == country_code]
country_terrestre = world_terrestre[world_terrestre["iso_3166_1_alpha_2_codes"] == "FR"]
country_maritime = world_maritime[world_maritime[critere_maritime] == country_code]

# Fusion des géométries terrestres et maritimes
geom_terrestre = country_terrestre.geometry.union_all()
geom_maritime = country_maritime.geometry.union_all() if not country_maritime.empty else None

if geom_maritime is not None:
    geom_fusionnee = geom_terrestre.union(geom_maritime)
else:
    geom_fusionnee = geom_terrestre

# Calcul du centre de latitude pour ajuster la grille
minx, miny, maxx, maxy = geom_terrestre.bounds
midpoint_lat = (miny + maxy) / 2

# 2. Fonction de création de grille avec renommage intégré
def create_and_rename_grid(source_gdf, country_code, column_name, grid_size_km, cle_geo, crop,display):
    grid = create_country_grid_WGS84(source_gdf, country_code, column_name, grid_size_km, midpoint_lat, crop, display=display)
    return grid.rename(columns={'cell_name': cle_geo})

# 3. Création des grilles terrestre, maritime et combinée
country_grid_terrestre = create_and_rename_grid(country_terrestre, country_code, "color_code", grid_size_km, cle_geo, crop=True,display=False)
country_grid_maritime = gpd.GeoDataFrame()

if not country_maritime.empty:
    country_grid_maritime = create_and_rename_grid(country_maritime, country_code, "ISO_TER1", grid_size_km, cle_geo, crop=True,display=False)

# Création d'un GeoDataFrame pour la géométrie fusionnée
gdf_fusionne = gpd.GeoDataFrame(geometry=[geom_fusionnee], crs=world_terrestre.crs)
gdf_fusionne["Code"] = country_code

# Création de la grille combinée
country_grid_combined = create_and_rename_grid(gdf_fusionne, country_code, "Code", grid_size_km, cle_geo, crop=False,display=False)

#Rendre les nom de mailles uniques
make_geo_keys_unique(country_grid_terrestre, cle_geo)
make_geo_keys_unique(country_grid_maritime, cle_geo)
make_geo_keys_unique(country_grid_combined, cle_geo)

#Vérification s'il y a des doublons
check_duplicates(country_grid_terrestre, "country_grid_terrestre", cle_geo)
check_duplicates(country_grid_maritime, "country_grid_maritime", cle_geo)
check_duplicates(country_grid_combined, "country_grid_combined", cle_geo)

# 5. Sauvegarde des fichiers
country_grid_terrestre.to_file(generate_file_path(country_name, grid_size_km, 'terrestre', cle_geo), driver="GeoJSON")

if not country_grid_maritime.empty:
    country_grid_maritime.to_file(generate_file_path(country_name, grid_size_km, 'maritime', cle_geo), driver="GeoJSON")

country_grid_combined.to_file(generate_file_path(country_name, grid_size_km, 'combined', cle_geo), driver="GeoJSON")


✅ Aucun doublon trouvé dans country_grid_terrestre.
✅ Aucun doublon trouvé dans country_grid_maritime.
✅ Aucun doublon trouvé dans country_grid_combined.


## TRAITEMENT DES DONNEES

In [59]:
# 📌 Paramètres généraux
#country_name = "France"
#grid_size_km = 20
cle_ID = "speciesKey"
#cle_geo = f"codeMaille{grid_size_km}Km"
path = path_data + rf"\GBIF_{country_name.replace(' ', '_')}"
bornes_temporelles = [1800, 1990, 2010, 2024]


In [60]:
# Générer le chemin des fichiers grid
path_grid_terrestre = generate_file_path(country_name, grid_size_km, 'terrestre', cle_geo)
path_grid_maritime = generate_file_path(country_name, grid_size_km, 'maritime', cle_geo)
path_grid_combined = generate_file_path(country_name, grid_size_km, 'combined', cle_geo)

# Vérifier si les fichiers grid existent et afficher un message approprié
def load_grid_file(path, name):
    if os.path.exists(path):
        print(f"✅ {name} trouvée et chargée avec succès !")
        return gpd.read_file(path)
    else:
        print(f"⚠️ {name} non trouvée. Elle ne sera pas traitée.")
        return None

country_grid_terrestre = load_grid_file(path_grid_terrestre, "Grid Terrestre")
country_grid_maritime = load_grid_file(path_grid_maritime, "Grid Maritime")
country_grid_combined = load_grid_file(path_grid_combined, "Grid Combined")


✅ Grid Terrestre trouvée et chargée avec succès !
✅ Grid Maritime trouvée et chargée avec succès !
✅ Grid Combined trouvée et chargée avec succès !


In [61]:
def process_biodiv_data(df,annee_min=1):
    """Nettoie et formate les données de biodiversité."""
    
    # Copier les données pour éviter les modifications sur l'original
    df_cleaned = df.copy()
    
    # Identifier le nombre initial d'espèces et d'observations
    n_especes_entrée = len(df_cleaned[cle_ID].unique())
    n_obs_entrée = len(df_cleaned)
    
    # 🔹 Convertir 'eventDate' en datetime et compléter 'year'
    df_cleaned['eventDate'] = pd.to_datetime(df_cleaned['eventDate'], errors='coerce', utc=True)
    df_cleaned['year'] = df_cleaned['year'].fillna(df_cleaned['eventDate'].dt.year)
    
    # 🔹 Assurez-vous que les coordonnées sont numériques et filtrer les NaN
    df_cleaned['decimalLongitude'] = pd.to_numeric(df_cleaned['decimalLongitude'], errors='coerce')
    df_cleaned['decimalLatitude'] = pd.to_numeric(df_cleaned['decimalLatitude'], errors='coerce')
    df_cleaned = df_cleaned.dropna(subset=['decimalLongitude', 'decimalLatitude', cle_ID]).reset_index(drop=True)

    print(f"➡️  En entrée : {n_especes_entrée} espèces, {n_obs_entrée} observations")

    # Suppression des lignes avec valeurs manquantes pour les colonnes cruciales
    df_cleaned = df_cleaned.dropna(subset=[cle_ID]).reset_index(drop=True)

    # Filtrer les observations où 'occurrenceStatus' est 'PRESENT'
    df_cleaned = df_cleaned[df_cleaned['occurrenceStatus'] == 'PRESENT'].reset_index(drop=True)

    # Convertir l'ID des espèces en entier
    df_cleaned[cle_ID] = df_cleaned[cle_ID].astype(int)
    
    df_cleaned['year'] = pd.to_numeric(df_cleaned['year'], errors='coerce').fillna(0).astype(int)
    if annee_min is not None:
        df_cleaned = df_cleaned[df_cleaned['year'] >= annee_min]

    # Renommer la colonne contenant la maille géographique
    df_cleaned.rename(columns={'grid_name': cle_geo}, inplace=True)

    # Convertir 'individualCount' en numérique et remplacer NaN par 1
    df_cleaned['individualCount'] = pd.to_numeric(df_cleaned['individualCount'], errors='coerce').fillna(1)

    # Calcul des pertes en pourcentage
    perte_especes = 100 - round(len(df_cleaned[cle_ID].unique()) / n_especes_entrée * 100)
    perte_obs = 100 - round(len(df_cleaned) / n_obs_entrée * 100)

    print(f"✅ En sortie : {len(df_cleaned[cle_ID].unique())} espèces (-{perte_especes}%)")
    print(f"✅ En sortie : {len(df_cleaned)} observations (-{perte_obs}%)\n")

    return df_cleaned

def add_grid_to_country(df_country, grid,cle_geo):
    df_new=df_country.copy()
    """
    Optimized version of adding the corresponding grid cell to each row in df_country based on latitude and longitude.
    
    Parameters:
    - df_country: DataFrame containing the columns 'decimalLatitude' and 'decimalLongitude'.
    - grid: DataFrame containing the grid cells with 'name', 'min_lon', 'min_lat', 'max_lon', and 'max_lat' columns.
    
    Returns:
    - df_country: Updated DataFrame with an additional 'grid_name' column indicating the grid cell for each point.
    """
    # Convert grid bounds to NumPy arrays for efficient vectorized comparison
    min_lons = grid['min_lon'].values
    max_lons = grid['max_lon'].values
    min_lats = grid['min_lat'].values
    max_lats = grid['max_lat'].values
    grid_names = grid[cle_geo].values
    
    # Initialize an array to store the grid names
    grid_names_for_points = []
    
    # Iterate over each point in df_country and apply vectorized comparison
    for lon, lat in zip(df_country['decimalLongitude'], df_country['decimalLatitude']):
        # Find the grid cell by comparing the point coordinates with grid bounds
        matching_grid = np.where((min_lons <= lon) & (lon <= max_lons) & (min_lats <= lat) & (lat <= max_lats))[0]
        
        if matching_grid.size > 0:
            grid_names_for_points.append(grid_names[matching_grid[0]])  # Take the first matching grid cell
        else:
            grid_names_for_points.append(None)  # No matching grid
    
    # Add the grid names to the DataFrame
    df_new[cle_geo] = grid_names_for_points
    
    return df_new
    
def formater_maille_espece_GBIF(df,cle_geo='codeMaille10Km',cle_ID='cdRef',bornes_temporelles=None):
    df_dico=generer_dictionnaire_taxonomie(df,cle_ID)
    # Convertir la colonne 'year' en int
    df['year'] = pd.to_numeric(df['year'], errors='coerce').fillna(0).astype(int)

    # Choisir des bornes temporelles et assigner une période aux données
    if bornes_temporelles is not None:
        df.loc[:, 'periode'] = pd.cut(df['year'], bins=bornes_temporelles, 
                       labels=[f'Période {i+1}: {bornes_temporelles[i]+1} à {bornes_temporelles[i+1]}' for i in range(len(bornes_temporelles) - 1)],
                       include_lowest=False)  # include_lowest=True inclut la borne inférieureure
        # Compter le nombre de données dans chaque intervalle
        compte_par_periode = df['periode'].value_counts()
        print(compte_par_periode)
         # Compter les occurrences d'observation de chaque taxon pour chaque code et période
        df_maille_espece = df.groupby([cle_geo, cle_ID,'periode'], observed=True).size().reset_index(name='nombreObs')
        #df_maille_espece = df.groupby([cle_geo, cle_ID,'periode'], observed=True)['individualCount'].sum().reset_index(name='nombreObs')
        #eventuellement remplacer ['individualCount'].sum() par .size() 
       
    else:
        # Compter les occurrences d'observation de chaque taxon pour chaque code
        df_maille_espece = df.groupby([cle_geo, cle_ID], observed=True).size().reset_index(name='nombreObs')
        #eventuellement remplacer ['individualCount'].sum() par .size() 
        
    df_maille_espece=pd.merge(df_maille_espece,df_dico,on=cle_ID)
    
    return df_maille_espece
    
# 📌 Fonction pour charger et traiter chaque chunk
def process_chunk(df_biodiv, country_grid_terrestre, country_grid_maritime, country_grid_combined, bornes_temporelles, chunk_number):
    """Prétraitement et enregistrement des données par chunk"""
    
    print(f"\n🔍 Traitement du chunk {chunk_number}...\n")

    df_biodiv = process_biodiv_data(df_biodiv)

    # 🔹 Ajouter la maille pour chaque catégorie
    if country_grid_terrestre is not None:
        df_terrestre = add_grid_to_country(df_biodiv, country_grid_terrestre, cle_geo).dropna(subset=[cle_geo]).reset_index(drop=True)
        
    if country_grid_maritime is not None:
        df_maritime = add_grid_to_country(df_biodiv, country_grid_maritime, cle_geo).dropna(subset=[cle_geo]).reset_index(drop=True)

    if country_grid_combined is not None:
        df_combined = add_grid_to_country(df_biodiv, country_grid_combined, cle_geo).dropna(subset=[cle_geo]).reset_index(drop=True)

    # 🔹 Regrouper par maille et période
    chaine_bornes = "_".join(map(str, bornes_temporelles))
    
    if country_grid_terrestre is not None:
        df_maille_espece_terrestre = formater_maille_espece_GBIF(df_terrestre, cle_geo, cle_ID, bornes_temporelles)
    if country_grid_maritime is not None:
        df_maille_espece_maritime = formater_maille_espece_GBIF(df_maritime, cle_geo, cle_ID, bornes_temporelles)
    if country_grid_combined is not None:
        df_maille_espece_combined = formater_maille_espece_GBIF(df_combined, cle_geo, cle_ID, bornes_temporelles)

    # 🔹 Ajouter les noms vernaculaires
    dico_noms_vernaculaires = pd.read_csv(path_data + r"\TAXO_GBIF\dico_noms_vernaculaires_merged.csv")

    if country_grid_terrestre is not None:
        df_maille_espece_terrestre = pd.merge(df_maille_espece_terrestre, dico_noms_vernaculaires, on=cle_ID, how="left")
    if country_grid_maritime is not None:
        df_maille_espece_maritime = pd.merge(df_maille_espece_maritime, dico_noms_vernaculaires, on=cle_ID, how="left")
    if country_grid_combined is not None:
        df_maille_espece_combined = pd.merge(df_maille_espece_combined, dico_noms_vernaculaires, on=cle_ID, how="left")
    
    # 🔹 Sauvegarde des fichiers
    base_path = path + "/"
    
    def save_chunk(df, ecosysteme):
        df.to_csv(base_path +'processed/'+ f"data_GBIF_{country_name.replace(' ', '_')}_{ecosysteme}_{cle_geo}_{cle_ID}_periodes{chaine_bornes}_{chunk_number}.csv", index=False)

    if country_grid_terrestre is not None:
        save_chunk(df_maille_espece_terrestre, "terrestre")
    if country_grid_maritime is not None:
        save_chunk(df_maille_espece_maritime, "maritime")
    if country_grid_combined is not None:
        save_chunk(df_maille_espece_combined, "combined")

    # 🔹 Nettoyage mémoire
    del df_biodiv, df_terrestre, df_maritime, df_combined
    del df_maille_espece_terrestre, df_maille_espece_maritime, df_maille_espece_combined
    print(f"\n✅ Chunk {chunk_number} traité et sauvegardé avec succès ! 🎉\n")

In [62]:
# 📌 Lecture et découpage du fichier
fichier = path_data + f"/GBIF_{country_name.replace(' ', '_')}/Raw/extractGBIF_{country_name.replace(' ', '_')}_12112024.csv"
colonnes_a_importer = ['kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'verbatimScientificName',
                        'taxonRank', 'countryCode', 'occurrenceStatus', 'individualCount', 
                        'decimalLongitude', 'decimalLatitude', 'eventDate', 'speciesKey', 'occurrenceID', 'year']
lines_per_chunk = 10_000_000
chunk_number = 0

# 📌 Ajouter une barre de progression pour le traitement des chunks:
for df_biodiv in pd.read_csv(fichier, sep='\t', chunksize=lines_per_chunk, on_bad_lines='skip', usecols=colonnes_a_importer):
    chunk_number += 1
    process_chunk(df_biodiv, country_grid_terrestre, country_grid_maritime, country_grid_combined, bornes_temporelles, chunk_number)
        
print(f"\n✅ Tous les chunks ont été traités avec succès. Nombre total de chunks traités : {chunk_number}")


C:\Users\User 1\AppData\Local\Temp\ipykernel_3116\2780776474.py:10: DtypeWarning: Columns (2,5,13,19,21,29,32) have mixed types. Specify dtype option on import or set low_memory=False.
  for df_biodiv in pd.read_csv(fichier, sep='\t', chunksize=lines_per_chunk, on_bad_lines='skip', usecols=colonnes_a_importer):



🔍 Traitement du chunk 1...

➡️  En entrée : 49327 espèces, 10000000 observations
✅ En sortie : 34976 espèces (-29%)
✅ En sortie : 6165574 observations (-38%)

periode
Période 3: 2011 à 2024    3979574
Période 2: 1991 à 2010    1588961
Période 1: 1801 à 1990     461994
Name: count, dtype: int64
periode
Période 3: 2011 à 2024    1148388
Période 2: 1991 à 2010     517461
Période 1: 1801 à 1990     158344
Name: count, dtype: int64
periode
Période 3: 2011 à 2024    3994913
Période 2: 1991 à 2010    1631888
Période 1: 1801 à 1990     486235
Name: count, dtype: int64

✅ Chunk 1 traité et sauvegardé avec succès ! 🎉



C:\Users\User 1\AppData\Local\Temp\ipykernel_3116\2780776474.py:10: DtypeWarning: Columns (2,6,29) have mixed types. Specify dtype option on import or set low_memory=False.
  for df_biodiv in pd.read_csv(fichier, sep='\t', chunksize=lines_per_chunk, on_bad_lines='skip', usecols=colonnes_a_importer):



🔍 Traitement du chunk 2...

➡️  En entrée : 40060 espèces, 10000000 observations
✅ En sortie : 30100 espèces (-25%)
✅ En sortie : 6271571 observations (-37%)

periode
Période 3: 2011 à 2024    4949646
Période 2: 1991 à 2010     978137
Période 1: 1801 à 1990     283944
Name: count, dtype: int64
periode
Période 3: 2011 à 2024    1577086
Période 2: 1991 à 2010     341520
Période 1: 1801 à 1990      84503
Name: count, dtype: int64
periode
Période 3: 2011 à 2024    4964384
Période 2: 1991 à 2010     996806
Période 1: 1801 à 1990     288000
Name: count, dtype: int64

✅ Chunk 2 traité et sauvegardé avec succès ! 🎉


🔍 Traitement du chunk 3...

➡️  En entrée : 19878 espèces, 10000000 observations
✅ En sortie : 19350 espèces (-3%)
✅ En sortie : 9923182 observations (-1%)

periode
Période 3: 2011 à 2024    9359093
Période 2: 1991 à 2010     458762
Période 1: 1801 à 1990      76892
Name: count, dtype: int64
periode
Période 3: 2011 à 2024    3098796
Période 2: 1991 à 2010     167989
Période 1: 18

C:\Users\User 1\AppData\Local\Temp\ipykernel_3116\2780776474.py:10: DtypeWarning: Columns (9,13,29) have mixed types. Specify dtype option on import or set low_memory=False.
  for df_biodiv in pd.read_csv(fichier, sep='\t', chunksize=lines_per_chunk, on_bad_lines='skip', usecols=colonnes_a_importer):


...


🔍 Traitement du chunk 7...

➡️  En entrée : 52280 espèces, 10000000 observations
✅ En sortie : 40249 espèces (-23%)
✅ En sortie : 6580919 observations (-34%)

periode
Période 3: 2011 à 2024    2865787
Période 2: 1991 à 2010    2759880
Période 1: 1801 à 1990     816823
Name: count, dtype: int64
periode
Période 2: 1991 à 2010    885621
Période 3: 2011 à 2024    864419
Période 1: 1801 à 1990    200561
Name: count, dtype: int64
periode
Période 3: 2011 à 2024    2879997
Période 2: 1991 à 2010    2812840
Période 1: 1801 à 1990     827140
Name: count, dtype: int64

✅ Chunk 7 traité et sauvegardé avec succès ! 🎉



C:\Users\User 1\AppData\Local\Temp\ipykernel_3116\2780776474.py:10: DtypeWarning: Columns (2,29) have mixed types. Specify dtype option on import or set low_memory=False.
  for df_biodiv in pd.read_csv(fichier, sep='\t', chunksize=lines_per_chunk, on_bad_lines='skip', usecols=colonnes_a_importer):



🔍 Traitement du chunk 8...



C:\Users\User 1\AppData\Local\Temp\ipykernel_3116\1988000803.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_cleaned['eventDate'] = pd.to_datetime(df_cleaned['eventDate'], errors='coerce', utc=True)


➡️  En entrée : 42474 espèces, 4702847 observations
✅ En sortie : 32102 espèces (-24%)
✅ En sortie : 2825645 observations (-40%)

periode
Période 3: 2011 à 2024    1539276
Période 2: 1991 à 2010     915127
Période 1: 1801 à 1990     309418
Name: count, dtype: int64
periode
Période 3: 2011 à 2024    469408
Période 2: 1991 à 2010    250899
Période 1: 1801 à 1990     94114
Name: count, dtype: int64
periode
Période 3: 2011 à 2024    1548102
Période 2: 1991 à 2010     942351
Période 1: 1801 à 1990     323382
Name: count, dtype: int64

✅ Chunk 8 traité et sauvegardé avec succès ! 🎉


✅ Tous les chunks ont été traités avec succès. Nombre total de chunks traités : 8


In [ ]:
# 📌 Concaténation des fichiers par écosystème
# 🔹 Regrouper par maille et période

# 📌 Vérifier et créer le dossier processed
processed_path = os.path.join(path, "processed")
os.makedirs(processed_path, exist_ok=True)  # Crée le dossier s'il n'existe pas

chaine_bornes = "_".join(map(str, bornes_temporelles))

for ecosysteme in ['maritime', 'combined', 'terrestre']:
    print(f"\n📢 Fusion des fichiers pour l'écosystème : {ecosysteme}")

    df_final = pd.DataFrame()

    for i in range(1, chunk_number+1):
        file_path = path +'/processed'+ f"/data_GBIF_{country_name.replace(' ', '_')}_{ecosysteme}_{cle_geo}_{cle_ID}_periodes{chaine_bornes}_{i}.csv"
        if os.path.exists(file_path):
            df_temp = pd.read_csv(file_path)
            df_final = pd.concat([df_final, df_temp], ignore_index=True)

    # 🔹 Regroupement des observations
    df_maille_espece = df_final.groupby([cle_geo, cle_ID, 'periode'], observed=True)['nombreObs'].sum().reset_index()

    # 🔹 Génération du dictionnaire taxonomique
    df_dico = generer_dictionnaire_taxonomie(df_final, cle_ID)

    # 🔹 Fusion des données
    df_maille_espece = pd.merge(df_maille_espece, df_dico, on=cle_ID)

    # 🔹 Sauvegarde du fichier fusionné
    final_file_path = path +'/processed'+ f"/data_GBIF_{country_name.replace(' ', '_')}_{ecosysteme}_{cle_geo}_{cle_ID}_periodes{chaine_bornes}.csv"
    df_maille_espece.to_csv(final_file_path, index=False)
    
    print(f"✅ Fichier fusionné sauvegardé : {final_file_path}")

    # 🔥 Suppression des fichiers chunk après fusion
    for i in range(1, chunk_number + 1):
        file_path = path +'/processed'+ f"/data_GBIF_{country_name.replace(' ', '_')}_{ecosysteme}_{cle_geo}_{cle_ID}_periodes{chaine_bornes}_{i}.csv"
        if os.path.exists(file_path):
            os.remove(file_path)
            print(f"🗑️  Supprimé : {file_path}")

print("\n🎉")


📢 Fusion des fichiers pour l'écosystème : maritime
✅ Fichier fusionné sauvegardé : D:\MANTIS\Data\GBIF_Spain/processed/data_GBIF_Spain_maritime_codeMaille20Km_speciesKey_periodes1800_1990_2010_2024.csv
🗑️  Supprimé : D:\MANTIS\Data\GBIF_Spain/processed/data_GBIF_Spain_maritime_codeMaille20Km_speciesKey_periodes1800_1990_2010_2024_1.csv
🗑️  Supprimé : D:\MANTIS\Data\GBIF_Spain/processed/data_GBIF_Spain_maritime_codeMaille20Km_speciesKey_periodes1800_1990_2010_2024_2.csv
🗑️  Supprimé : D:\MANTIS\Data\GBIF_Spain/processed/data_GBIF_Spain_maritime_codeMaille20Km_speciesKey_periodes1800_1990_2010_2024_3.csv
🗑️  Supprimé : D:\MANTIS\Data\GBIF_Spain/processed/data_GBIF_Spain_maritime_codeMaille20Km_speciesKey_periodes1800_1990_2010_2024_4.csv
🗑️  Supprimé : D:\MANTIS\Data\GBIF_Spain/processed/data_GBIF_Spain_maritime_codeMaille20Km_speciesKey_periodes1800_1990_2010_2024_5.csv
🗑️  Supprimé : D:\MANTIS\Data\GBIF_Spain/processed/data_GBIF_Spain_maritime_codeMaille20Km_speciesKey_periodes1800_19